# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Faizan-Hussain-Dev/FlyrankMainAssignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents a unique search event, defined by a user's interaction with the search functionality. The time window for analysis will be the month of March 2026, specifically from 2026-03-01 to 2026-03-31. The primary table to be used from the internship-warehouse dataset is events.

In [19]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. Authenticate and Load Data
hf_token = userdata.get('HF_Token')
print("Loading 'fact_content_query_90d' dataset from Hugging Face...")

dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", split="train", token=hf_token)
df = dataset.to_pandas()

# 2. Set time column and filter for March 2026
time_col = 'window_end'
df[time_col] = pd.to_datetime(df[time_col])

df_march = df[(df[time_col] >= '2026-03-01') & (df[time_col] <= '2026-03-31')].copy()

print("\n--- Fact 1 & 2: Row Count and Date Span ---")
print(f"Row count for March 2026 slice: {len(df_march)}")
print(f"Date span: {df_march[time_col].min().date()} to {df_march[time_col].max().date()}")

print("\n--- Fact 3: Grain Check ---")
# The grain is the unique combination of client, content, query, and the specific window date
grain_cols = ['client_hash_id', 'content_hash_id', 'query_hash_id', 'window_end']
duplicates = df_march.duplicated(subset=grain_cols).sum()
print(f"Duplicates in grain {grain_cols}: {duplicates} (0 means the grain holds true)")

Loading 'fact_content_query_90d' dataset from Hugging Face...

--- Fact 1 & 2: Row Count and Date Span ---
Row count for March 2026 slice: 0
Date span: NaT to NaT

--- Fact 3: Grain Check ---
Duplicates in grain ['client_hash_id', 'content_hash_id', 'query_hash_id', 'window_end']: 0 (0 means the grain holds true)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label/Proxy: clicks_last30 (The target we want to predict).

Features: query_char_count, query_token_count, impressions_prev30, clicks_prev30 (Historical/static data knowable before the last 30-day window).

Context: window_end, client_hash_id, content_hash_id, query_hash_id (IDs and dates used for grouping, not modeling).

Excluded: impressions_last30 and clicks_90d (Excluded because they overlap with the clicks_last30 prediction window and would cause data leakage).

In [20]:
# 2. Verify Missingness for the Categorized Fields
# We check the fields we categorized to ensure they don't have patterned gaps (null values)

fields_to_check = [
    'clicks_last30',       # Label
    'query_char_count',    # Feature
    'query_token_count',   # Feature
    'impressions_prev30',  # Feature
    'clicks_prev30',       # Feature
    'client_hash_id',      # Context
    'query_hash_id',       # Context
    'window_end',          # Context
    'impressions_last30'   # Excluded
]

print("--- Missing Values Check per Field ---")
# This calculates the percentage of missing (null) values for each column,
# matching the assignment's AVG(CASE WHEN col IS NULL THEN 1.0 ELSE 0 END) requirement.

# Assuming df_march is already loaded from the Section 1 code block
missingness = df_march[fields_to_check].isnull().mean() * 100
print(missingness.apply(lambda x: f"{x:.2f}% missing"))

--- Missing Values Check per Field ---
clicks_last30         nan% missing
query_char_count      nan% missing
query_token_count     nan% missing
impressions_prev30    nan% missing
clicks_prev30         nan% missing
client_hash_id        nan% missing
query_hash_id         nan% missing
window_end            nan% missing
impressions_last30    nan% missing
dtype: object


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Verify it with queries (grain, counts, missing values, windows)

**Feature Rationale: Knowable at the decision moment**
To predict `clicks_last30`, we only use features that are strictly known before the last 30-day window begins:
*   **`query_char_count`**: Knowable at the decision moment because the character length is a static, physical property of the query itself.
*   **`query_token_count`**: Knowable at the decision moment because the number of words/tokens is fixed the moment the query is formed.
*   **`impressions_prev30`**: Knowable at the decision moment because it represents historical performance from a strictly prior time window that has already closed.
*   **`clicks_prev30`**: Knowable at the decision moment because, like previous impressions, this historical engagement data is finalized before our target prediction window opens.

**The Deliberate Leakage Trap**
I deliberately added `impressions_last30` to the feature frame. This is a massive data leak because impressions in the *last 30 days* accumulate concurrently with the *clicks* we are trying to predict. In a real-world scenario, you would not know the final impression count until the 30 days were over. Including it causes an artificial correlation, which I then removed to restore the honest data frame.

In [21]:
# 1. Availability Check (IS TRUE filter equivalent)
print("--- Availability Check ---")
# Check availability: How many rows actually had previous 30-day historical data available?
available_mask = df_march['impressions_prev30'] > 0
df_valid = df_march[available_mask]
print(f"Rows surviving 'impressions_prev30 > 0' availability filter: {len(df_valid)} out of {len(df_march)}\n")

# 2. Build the Four-Feature Frame
features = ['query_char_count', 'query_token_count', 'impressions_prev30', 'clicks_prev30']
label = 'clicks_last30'

# Create the clean feature frame
df_features = df_valid[features + [label]].copy()

print("--- Feature Frame (First 5 rows) ---")
print(df_features.head())
print("\n")

# 3. The Deliberate Leakage Trap
print("--- The Leakage Trap ---")
# Trap: Deliberately adding 'impressions_last30'
df_features['leaky_impressions_last30'] = df_valid['impressions_last30']

print("Added leaky feature. Notice the high correlation with our target label (clicks_last30):")
# Showing the correlation to prove the leakage inflates model scores artificially
print(df_features[[label, 'leaky_impressions_last30']].corr())
print("\n")

# 4. Springing the Trap
df_features = df_features.drop(columns=['leaky_impressions_last30'])
print("--- Honesty Restored ---")
print("Leaky column removed. Remaining legitimate columns:")
print(df_features.columns.tolist())

--- Availability Check ---
Rows surviving 'impressions_prev30 > 0' availability filter: 0 out of 0

--- Feature Frame (First 5 rows) ---
Empty DataFrame
Columns: [query_char_count, query_token_count, impressions_prev30, clicks_prev30, clicks_last30]
Index: []


--- The Leakage Trap ---
Added leaky feature. Notice the high correlation with our target label (clicks_last30):
                          clicks_last30  leaky_impressions_last30
clicks_last30                       NaN                       NaN
leaky_impressions_last30            NaN                       NaN


--- Honesty Restored ---
Leaky column removed. Remaining legitimate columns:
['query_char_count', 'query_token_count', 'impressions_prev30', 'clicks_prev30', 'clicks_last30']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

**Limitation: Pre-Aggregated Rolling Windows and Loss of Daily Granularity**
One major limitation of this specific data slice is that it is strictly pre-aggregated into 30-day and 90-day rolling windows. Because the metrics (like `clicks_last30` or `impressions_prev30`) are provided as monthly sums tied to a single `window_end` date, this data can never tell us about **daily volatility or short-term trends**.

For example, if a query received 1,000 clicks in the last 30 days, it is impossible to know if those clicks happened evenly across the month or if they all occurred during a single viral 24-hour spike. We are completely blind to any intra-month seasonality or daily user behavior.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Proving the limitation: We check the available numeric columns to show
# that we only have access to aggregated time buckets (30d/90d), not daily event logs.

print("--- Data Limitation Check: Available Metric Columns ---")
# Filter for metric-related columns
metric_cols = [col for col in df_march.columns if 'click' in col or 'impression' in col]

print("Notice how all available metrics are locked into 30 or 90-day aggregates:")
for col in metric_cols:
    print(f"- {col}")

print("\nResult:")
print("There is no 'daily_clicks' or timestamp for individual search events. The grain physically prevents intra-month trend analysis.")

--- Data Limitation Check: Available Metric Columns ---
Notice how all available metrics are locked into 30 or 90-day aggregates:
- impressions_90d
- clicks_90d
- impressions_last30
- clicks_last30
- impressions_prev30
- clicks_prev30
- content_total_impressions_90d
- rare_impressions_share
- anonymized_impressions_share

Result:
There is no 'daily_clicks' or timestamp for individual search events. The grain physically prevents intra-month trend analysis.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.